# 5 — Blueprint et soumission Kaggle (CRISP-DM Phase 6)

Ce notebook est le **rapport à la direction Inved Corp** (synthèse exécutive + plan de production) **et** le générateur de la soumission Kaggle. La Phase 6 ne **rejuge pas** le modèle — la Phase 5 (NB4) l'a fait — elle part de son verdict et répond à une autre question : *« le modèle reste-t-il bon une fois en production ? »*.

---

## Synthèse exécutive — rapport à la direction

**Contexte métier.** Inved Corp veut un **outil assistif d'estimation** pour ses consultants : sur soumission d'un formulaire, l'outil renvoie un **prix estimé + intervalle de confiance + top-3 facteurs explicatifs**, le consultant gardant la **validation finale**. Deux usages : le **conseil rénovation** (quand l'estimation IA < attente client, on montre les leviers actionnables, ex. `KitchenQual`) et le label **« Prix certifié par Inved AI »** (quand IA ≈ marché). KPIs visés : temps d'expertise **4 h → 2 h**, **+20 %** de ventes sous 90 jours, **< 10 %** d'override manuel.

**Modèle final (verdict Phase 5, non rejugé ici).** Champion = **Stacking** (Lasso + GradientBoosting + XGBoost → méta-RidgeCV), défendu sur les trois axes :
- **Mathématique** — RMSLE holdout **0,1122** (OOF 0,111), mais en **quasi ex-æquo statistique** (IC bootstrap chevauchant, NB4 §5.2.1) ; sa **diversité d'erreurs** fait la différence.
- **Informatique** — latence **~56 ms/requête** (≈ 90× sous le SLA de 5 s) ; son vrai coût est la **surface de monitoring** (4 sous-modèles), pas le calcul.
- **Métier** — interprétabilité indirecte (proxy SHAP) ; **équité** correcte (résidus par quartier dans ≈ [−4 %, +6 %]).

**Recommandation.** Livrer le **Stacking** pour le label « certifié ». **Repli légitime** : **CatBoost** ou **XGBoost tuné** (Δ ≈ 0,005 RMSLE, dans le bruit), **1 seul artefact** à servir/monitorer, si la direction privilégie la simplicité d'exploitation.

**Déploiement (résumé).** API **temps réel** (réponse < 5 s sur soumission), **MLflow Model Registry → `mlflow models serve`** comme chemin d'inférence canonique, **réentraînement mensuel** (fenêtre glissante), garde-fou **RMSLE ≤ 0,13**.

**Risques majeurs.** Opacité du Stacking, **dérive du marché / gentrification**, équité sur les quartiers à faible volume. Détail ci-dessous.

---

## Sommaire du blueprint
- **Synthèse exécutive** (ci-dessus)
- **§6.1–6.4** — reconstruction du champion + soumission Kaggle (partie technique)
- **§6.5** — Stratégie de déploiement
- **§6.6** — Monitoring sur trois couches (= les 3 axes d'évaluation de la Phase 5, en continu)
- **§6.7** — Modèle de coût cloud
- **§6.8** — Cas d'école transverse : la gentrification
- **§6.9** — Risques et limites du champion

## 6.1 Setup et identification du champion

In [1]:
%load_ext autoreload
%autoreload 2
%run 2_data_prep.ipynb

Dimensions brutes : (1460, 81)
Nombre de points supprimés : 2
Dimensions après suppression : (1458, 81)
Variables après ingénierie : 90 colonnes (+8 dérivées)
Corrélation des variables dérivées avec SalePrice_log :
  TotalSF            : +0.825
  TotalBathrooms     : +0.677
  HouseAge           : -0.588
  YearsSinceRemodel  : -0.569
  GarageAge          : -0.543
  HasGarage          : +0.323
  HasSecondFloor     : +0.151
  HasPool            : +0.077   <-- faible (|r| < 0.1)
X_train : (1166, 83)   |   X_test : (292, 83)
Colonnes retirées (colinéarité) : ['GarageArea', 'TotalBsmtSF', 'TotRmsAbvGrd', 'GarageYrBlt']
NaN dans X_train : 6276 cellules sur 96778
Audit de cardinalité des colonnes nominales (sur X_train) :
Colonne           modalités   bucket
  Neighborhood           25     TargetEncoder (haute)  [imput. mode]
  Exterior2nd            16     TargetEncoder (haute)  [imput. mode]
  Exterior1st            15     TargetEncoder (haute)  [imput. mode]
  MSSubClass             15     

In [2]:
winner_path = RESULTS_DIR / 'winning_model.json'
if not winner_path.exists():
    raise RuntimeError(
        "results/winning_model.json absent. Exécuter 4_evaluation.ipynb d'abord."
    )

winner = json.loads(winner_path.read_text())
print(f"Modèle champion désigné par 4_evaluation : {winner['model']}  (famille : {winner['family']})")
print(f"  Holdout RMSLE : {winner['holdout_rmsle']:.4f}")
print(f"  Params        : {winner.get('params')}")

Modèle champion désigné par 4_evaluation : Stacking  (famille : stacking)
  Holdout RMSLE : 0.1122
  Params        : {'meta': 'RidgeCV', 'base': ['Lasso', 'GBR', 'XGB']}


## 6.2 Reconstruction et réentraînement du champion sur **toutes** les données d'entraînement

Pour la soumission Kaggle on entraîne sur 100 % des données disponibles (`X` complet, pas seulement `X_train` qui exclut 20 %). Le préprocesseur reste construit selon les règles définies dans `2_data_prep.ipynb`.

Le *dispatcher* ci-dessous mappe chaque nom de modèle vers la fonction qui le reconstruit. Quand on ajoutera LightGBM ou CatBoost, il suffira de rallonger ce dispatcher.

In [3]:
from sklearn.linear_model import LinearRegression, LassoCV, RidgeCV, ElasticNetCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, StackingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
# CatBoostRegressorCV + CatBoostPrep proviennent de 2_data_prep (via %run).


def _scaled(model):
    return Pipeline([('preprocessor', preprocessor_scaled), ('model', model)])

def _encoded(model):
    return Pipeline([('preprocessor', preprocessor_encoded), ('model', model)])

def _native(model):
    return Pipeline([('preprocessor', preprocessor_native), ('model', model)])


def build_champion(model_name: str, params: dict | None):
    p = params or {}

    if model_name == 'OLS':
        return _scaled(LinearRegression())
    if model_name == 'Ridge':
        return _scaled(RidgeCV(alphas=np.logspace(-2, 2, 20), cv=5))
    if model_name == 'Lasso':
        return _scaled(LassoCV(alphas=np.logspace(-4, 2, 20), cv=5, max_iter=50_000, random_state=RANDOM_STATE))
    if model_name == 'ElasticNet':
        return _scaled(ElasticNetCV(l1_ratio=[.1, .5, .7, .9, .95, .99, 1],
                                    alphas=np.logspace(-4, 2, 20), cv=5, max_iter=50_000, random_state=RANDOM_STATE))
    if model_name == 'KNN':
        return _scaled(KNeighborsRegressor(n_neighbors=int(p.get('k', 9))))
    if model_name == 'MLPRegressor':
        return _scaled(MLPRegressor(hidden_layer_sizes=(64,), activation='relu', solver='adam',
                                    max_iter=1500, early_stopping=True, validation_fraction=0.15,
                                    random_state=RANDOM_STATE))

    if model_name == 'DecisionTree':
        return _encoded(DecisionTreeRegressor(max_depth=int(p.get('max_depth', 6)), random_state=RANDOM_STATE))
    if model_name == 'RandomForest':
        return _encoded(RandomForestRegressor(n_estimators=int(p.get('n_estimators', 300)),
                                              random_state=RANDOM_STATE, n_jobs=-1))
    if model_name == 'GradientBoosting':
        return _encoded(GradientBoostingRegressor(
            n_estimators=int(p.get('n_estimators', 300)),
            learning_rate=float(p.get('learning_rate', 0.05)),
            max_depth=int(p.get('max_depth', 3)),
            random_state=RANDOM_STATE,
        ))
    if model_name == 'AdaBoost':
        return _encoded(AdaBoostRegressor(n_estimators=int(p.get('n_estimators', 200)), random_state=RANDOM_STATE))

    if model_name in {'XGBoost_native', 'XGBoost_onehot', 'XGBoost_tuned'}:
        defaults = dict(n_estimators=500, learning_rate=0.05, max_depth=4)
        cfg = {**defaults, **{k: v for k, v in p.items() if k in {
            'n_estimators', 'learning_rate', 'max_depth',
            'min_child_weight', 'subsample', 'colsample_bytree',
            'reg_alpha', 'reg_lambda',
        }}}
        # XGBoost_onehot used preprocessor_encoded; the others use preprocessor_native
        prep = preprocessor_encoded if model_name == 'XGBoost_onehot' else preprocessor_native
        kwargs = dict(random_state=RANDOM_STATE, n_jobs=-1, verbosity=0, tree_method='hist')
        if prep is preprocessor_native:
            kwargs['enable_categorical'] = True
        return Pipeline([('preprocessor', prep), ('model', XGBRegressor(**cfg, **kwargs))])

    if model_name == 'LightGBM':
        return _native(LGBMRegressor(
            n_estimators=int(p.get('n_estimators', 500)),
            learning_rate=float(p.get('learning_rate', 0.05)),
            num_leaves=int(p.get('num_leaves', 31)),
            random_state=RANDOM_STATE, n_jobs=-1, verbose=-1))

    if model_name == 'CatBoost':
        cat_cols = X_train.select_dtypes(include=['object', 'string']).columns.tolist()
        return Pipeline([
            ('preprocessor', CatBoostPrep()),
            ('model', CatBoostRegressorCV(
                cat_features=cat_cols,
                iterations=int(p.get('iterations', 500)),
                learning_rate=float(p.get('learning_rate', 0.05)),
                depth=int(p.get('depth', 6)),
                random_seed=RANDOM_STATE)),
        ])

    if model_name == 'Stacking':
        lasso_sub = _scaled(LassoCV(alphas=np.logspace(-4, 2, 20), cv=5, max_iter=50_000, random_state=RANDOM_STATE))
        gb_sub    = _encoded(GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE))
        xgb_sub   = _native(XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=4,
                                         enable_categorical=True, tree_method='hist',
                                         random_state=RANDOM_STATE, n_jobs=-1, verbosity=0))
        return StackingRegressor(
            estimators=[('lasso', lasso_sub), ('gbr', gb_sub), ('xgb', xgb_sub)],
            final_estimator=RidgeCV(alphas=np.logspace(-2, 2, 10)),
            cv=5, n_jobs=-1, passthrough=False,
        )

    raise ValueError(f"Modèle inconnu dans le dispatcher : {model_name}")


champion = build_champion(winner['model'], winner.get('params'))
print(f"Champion reconstruit : {type(champion).__name__}")

Champion reconstruit : StackingRegressor


## 6.3 Entraînement final sur 100 % des données

`X` et `y_log` sont définis par `2_data_prep.ipynb` (avant le `train_test_split`). On les utilise tels quels pour bénéficier de toutes les observations disponibles.

In [4]:
t0 = time.time()
champion.fit(X, y_log)
fit_s = time.time() - t0
print(f"Champion entraîné sur {len(X)} observations en {fit_s:.1f}s")

Champion entraîné sur 1458 observations en 9.2s


## 6.4 Application au jeu de test Kaggle

On charge `data/test.csv` (le **vrai** test set Kaggle, sans `SalePrice`), on applique exactement les mêmes drops de colonnes que sur le train (`Id`, `SalePrice*`), puis on prédit en espace log avant retransformation `expm1`.

In [5]:
df_test = pd.read_csv('./data/test.csv', sep=',')
test_ids = df_test['Id']
print(f"Test Kaggle : {df_test.shape}")

# Mêmes variables dérivées + cast MSSubClass que sur le train (fonction partagée du NB2).
# Le reindex sur X.columns applique aussi le retrait des colonnes colinéaires.
X_test_kaggle = engineer_features(df_test).drop(columns=['SalePrice', 'SalePrice_log', 'Id'], errors='ignore')
X_test_kaggle = X_test_kaggle[X.columns]

preds_log = champion.predict(X_test_kaggle)
preds_dollars = np.expm1(preds_log)

print(f"Statistiques des prédictions ($) :")
print(f"  min: {preds_dollars.min():,.0f}")
print(f"  mean: {preds_dollars.mean():,.0f}")
print(f"  max: {preds_dollars.max():,.0f}")
print(f"  nb NaN: {int(np.isnan(preds_dollars).sum())}")

assert not np.isnan(preds_dollars).any(), "Prédictions contiennent des NaN — modèle ou prétraitement défaillant"
assert (preds_dollars > 0).all(), "Prédictions <= 0 — anomalie"
assert len(preds_dollars) == len(df_test), "Longueur de prédictions ≠ longueur du test set"

submission = pd.DataFrame({'Id': test_ids, 'SalePrice': preds_dollars})
submission.to_csv('submission.csv', index=False)
print(f"\nSoumission écrite : submission.csv ({len(submission)} lignes)")
display(submission.head())

Test Kaggle : (1459, 80)
Statistiques des prédictions ($) :
  min: 46,443
  mean: 179,047
  max: 1,123,334
  nb NaN: 0

Soumission écrite : submission.csv (1459 lignes)


,Id,SalePrice
0,1461,117689.646470
1,1462,157865.292195
2,1463,180492.745364
3,1464,195888.388646
4,1465,190536.060290


## 6.5 Stratégie de déploiement

> *Architecture cible* (la mise en œuvre MLflow/POC est planifiée — cf. questions produit ouvertes). On décrit le plan, pas un système déjà en production.

**Mode d'exécution : temps réel.** L'usage Canvas est une **soumission de formulaire → réponse immédiate** : on sert donc le modèle en **synchrone**, sous le SLA de 5 s (la latence mesurée ~56 ms/requête laisse une marge confortable, NB4 §5.5.3). Un mode **batch** n'est utile que pour un cas secondaire (ré-estimation périodique d'un portefeuille), pas pour le flux principal.

**Chemin d'inférence canonique : MLflow.** Le modèle champion est enregistré dans le **MLflow Model Registry**, puis servi via **`mlflow models serve`** (API REST). Cela colle 1:1 à l'engagement Canvas (« API hébergée dans le cloud, CPU standard ») et rend le monitoring système (couche 2) *réel* (latence et taux d'erreur mesurés sur de vraies requêtes HTTP). L'artefact **joblib** persisté reste un **fallback** hors-ligne.

**Environnements & CI/CD.** Trois environnements **dev / test / prod**. Le pipeline CI/CD : (1) tests de **schéma** des entrées (colonnes, types), (2) **non-régression RMSLE** sur un holdout figé (refus de promotion si RMSLE > 0,13), (3) build de l'image, (4) promotion en *Staging* puis *Production* dans le Registry.

**Versioning & rollback.** Chaque réentraînement crée une **version** dans le Registry ; les *stages* (Staging/Production/Archived) permettent un **rollback immédiat** (repointer la version précédente) si une dérive est détectée. Les événements de déploiement alimentent la couche 2 du monitoring.

## 6.6 Monitoring — trois couches

**Le fil conducteur du livrable.** Les trois couches de monitoring **sont les trois axes d'évaluation de la Phase 5, projetés sur la production en continu** : la Phase 5 a jugé le modèle *une fois* (statique) pour le **choisir** ; la Phase 6 rebranche les mêmes axes *en permanence* pour le **surveiller**. Chaque métrique est rattachée à un engagement du ML Canvas.

### Couche 1 — Mathématique (le modèle se trompe-t-il davantage ?)
- **RMSLE sur prix réalisés** : recalculée dès qu'une vente se conclut, comparée au seuil **0,13** (NB4 §5.5.5).
- **Data drift** : PSI / KS sur les variables clés (`GrLivArea`, `TotalSF`, `Neighborhood`, `OverallQual`) entrée vs train.
- **Prediction drift** : dérive de la distribution des prix prédits.
- **Stabilité des importances SHAP** : un glissement du top-3 facteurs = signal de dérive de marché.
- **Déclencheurs de réentraînement** : cadence **mensuelle** *ou* RMSLE > 0,13 *ou* PSI au-delà d'un seuil.

### Couche 2 — Système / IT (l'API tient-elle ses promesses techniques ?)
- **Latence p50 / p95 / p99** vs SLA **5 s** (baseline ~56 ms/requête mesurée, NB4 §5.5.3) — mesurée sur les vraies requêtes HTTP via `mlflow models serve`.
- **Taux d'erreur** (5xx), **volume de requêtes**.
- **Logs de prédiction structurés**, rétention **90 jours** (alimente l'audit d'équité, couche 3).
- **Événements de déploiement** (`MlflowClient().search_model_versions`), **CPU / mémoire**.

### Couche 3 — Métier (l'outil crée-t-il la valeur promise ?)
- **KPIs du Canvas** : temps d'expertise **4 h → 2 h**, **+20 %** de ventes sous 90 jours, taux d'**override < 10 %**.
- **A/B test** de déploiement progressif : 3 mois, **50/50** vs expertise manuelle.
- **Audit d'équité géographique continu** (contrainte Canvas) — la version production de l'analyse OOF de NB4 §5.5.4.
- **Boucle de feedback consultant** via CRM (capteur précoce de dérive, cf. §6.8) et **satisfaction client** post-vente.
- **Impact du label « Prix certifié par Inved AI »** sur la conversion et le prix de vente.

## 6.7 Modèle de coût cloud

> Chiffres **estimés** (ordre de grandeur), sauf la latence — seule grandeur réellement mesurée (NB4 §5.5.3). Le but est le *raisonnement*, pas la facture exacte.

| Poste | Ordre de grandeur | Commentaire |
|---|---|---|
| **Entraînement** (compute) | ~nul | Réentraînement mensuel ~13 s sur CPU → quelques centimes/mois. |
| **Service API** (instance always-on) | **poste dominant** | Une instance CPU standard disponible 24/7 pour tenir le SLA < 5 s — c'est le vrai coût récurrent. |
| **Stockage + Registry** | négligeable | Artefacts modèles + logs ; quelques Go. |
| **Monitoring** (logs, dashboards, alertes) | modéré | Rétention 90 j des logs de prédiction (audit équité) + tableaux de bord. |
| **Validation consultant** | réel mais humain | Hors cloud : c'est du temps-homme, pas du compute. |

**Conclusion contre-intuitive.** Le **modèle est quasi gratuit** ; le coût récurrent réel est la **disponibilité de l'API + le monitoring + le workflow de validation humaine**. Le « surcoût » du **Stacking** face à un modèle unique n'est donc **pas le calcul** (latence négligeable face au SLA) mais sa **surface de monitoring** : 4 sous-modèles à surveiller pour la dérive plutôt qu'un seul. C'est l'argument économique du repli vers CatBoost / XGBoost tuné si la maintenabilité prime.

## 6.8 Cas d'école transverse : la gentrification

Ames abrite le campus de l'**Iowa State University** ; la pression étudiante peut **gentrifier** un quartier autrefois bon marché. C'est le scénario qui illustre le mieux pourquoi les **trois couches** sont nécessaires *ensemble* — il les frappe simultanément :

- **Couche 1 (mathématique) — double dérive.** *Data drift* : la distribution des prix d'un quartier se décale (PSI/KS sur `Neighborhood` × prix le détecte). *Concept drift* plus insidieux : la **relation features → prix change** (un même bien vaut soudain plus parce que le quartier est devenu prisé) — la dérive de prédiction et la chute de RMSLE sur ventes réalisées le révèlent.
- **Couche 3 (métier) — signal précoce.** Les **consultants corrigent à la hausse** systématiquement dans ce quartier *avant* que les ventes réalisées ne confirment la tendance : le taux et le sens des *overrides* (suivis via le CRM) sont un **détecteur avancé**, plus rapide que la métrique mathématique qui attend les transactions.
- **Tension d'équité — inverse du redlining.** Le modèle, entraîné sur l'**historique**, **sous-estime** un quartier en gentrification (il n'a pas encore « vu » la hausse). Ici l'équité ne consiste pas à éviter de sur-pénaliser un quartier pauvre (redlining classique) mais à **ne pas figer** un quartier en mutation dans son passé — un biais temporel, pas géographique.

**Réponse outillée.** C'est exactement ce que couvre le dispositif : **fenêtre glissante mensuelle** + **déclencheur de réentraînement** (cadence *ou* RMSLE > 0,13 *ou* dérive PSI), et la boucle consultant comme capteur précoce. La gentrification n'est donc pas un angle mort, mais le **cas d'usage qui justifie** l'architecture de monitoring à trois couches.

## 6.9 Risques et limites du champion

- **Opacité du Stacking.** Le modèle servi n'a **pas d'attribution directe** : le « top-3 facteurs » promis au consultant est calculé via **SHAP sur XGBoost** comme proxy lisible (3c §4.7), donc une **approximation** du raisonnement de l'ensemble — à présenter comme telle, et à valider humainement.
- **Pas d'extrapolation hors domaine.** Modèles à base d'arbres : un bien plus grand ou plus cher que tout l'historique voit sa prédiction **plafonnée** (cf. les plus grosses erreurs, NB4 §5.5.2.2). Les biens d'exception restent du ressort du consultant.
- **Équité sur quartiers à faible volume.** Le léger penchant à sous-estimer BrkSide / IDOTRR (~+5–6 %, NB4 §5.5.4) est un **signal mineur à suivre**, pas un biais systémique — mais il doit rester sous surveillance (couches 1 + 3).
- **Dérive du marché / gentrification.** Cf. §6.8 — le risque structurel principal, traité par la fenêtre glissante mensuelle + déclencheur de réentraînement.
- **Quasi ex-æquo (rappel).** Le Stacking n'est **pas significativement meilleur** que CatBoost / XGBoost tuné (IC chevauchants) ; un **repli vers un modèle unique** simplifierait l'exploitation (¼ de la surface de monitoring) pour une perte de RMSLE dans le bruit. Décision réversible, à réévaluer en production.

---

*Fin du blueprint. La Phase 5 a sélectionné le champion ; la Phase 6 l'a réentraîné sur 100 % des données, produit la soumission Kaggle, et défini comment le déployer, le surveiller (3 couches) et le faire évoluer en production.*